# SAMPIC Waveform Visualization

Comprehensive notebook for analyzing SAMPIC waveforms using PyROOT.
Compact SD no longer stores amplitude, baseline or peak. Here baseline = mean of the first 8 valid samples, amplitude = max − min, and peak = waveform maximum; these are notebook estimates, not vendor results. Calibrated samples are shown in mV.


In [ ]:
# Edit this path, then run the notebook from the top.
# Absolute paths and ~ are supported; relative paths start at the app directory.
DATA_FILE = ".artifacts/bank_format_verification/run00266/output.root"


In [ ]:
from pathlib import Path
import sys, os
# Works when Jupyter starts in the repository or a notebook subdirectory.
APP_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'notebooks/notebook_support.py').is_file())
if str(APP_ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(APP_ROOT / 'notebooks'))
from notebook_support import (open_data, present, hit_waveform, hit_baseline,
    hit_amplitude, hit_peak, waveform_entries as find_waveform_entries,
    sampic_unit, integer, waveform_packets, sampic_event_bytes)
import ROOT
r = ROOT
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
root_file, tree, DATA_FILE = open_data('SAMPIC', DATA_FILE)
f = file = root_file
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.25})
UNIT = sampic_unit(tree)
print('Waveform display units:', UNIT)


## Load Libraries and Data

In [ ]:
waveform_entries = find_waveform_entries(tree)
print(f'{len(waveform_entries)} entries contain nonempty waveforms')


## Explore Data Structure

In [ ]:
first_waveform_entry = waveform_entries[0] if waveform_entries else None
if first_waveform_entry is not None:
    tree.GetEntry(first_waveform_entry)
    for i, hit in enumerate(tree.sampic_event.hits):
        if i == 8: break
        print(f'Hit {i}: FEB={hit.fe_board_index}, channel={hit.channel}, samples={len(hit.corrected_waveform)}, '
              f'peak-to-peak={hit_amplitude(hit):.3f} {UNIT}, first cell={hit.first_cell_timestamp:.3f} ns')
else:
    print('No nonempty waveform entries in this file.')


## Individual Waveform Plots

In [ ]:
def plot_waveform(event_idx, hit_idx=0):
    """Plot a waveform by ROOT tree entry, safely handling trigger-only rows."""
    if event_idx is None: return None
    if event_idx < 0 or event_idx >= tree.GetEntries():
        raise IndexError(f'Entry {event_idx} is outside the tree')
    if event_idx is None: return None
    tree.GetEntry(event_idx)
    if tree.GetBranch('has_sampic_event') and not bool(tree.has_sampic_event):
        print(f'Entry {event_idx} contains trigger metadata but no waveform bank')
        return None
    event = tree.sampic_event
    if hit_idx < 0 or hit_idx >= len(event.hits):
        print(f'Hit {hit_idx} is outside entry {event_idx} ({len(event.hits)} hits)')
        return None

    hit = event.hits[hit_idx]
    wf = hit_waveform(hit)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(np.arange(wf.size), wf, linewidth=1.5, label='Corrected waveform')
    ax.set_xlabel('Sample index')
    ax.set_ylabel(f'Amplitude [{UNIT}]')
    ax.set_title(f'Entry {event_idx}, hit {hit_idx}, channel {hit.channel}')
    ax.legend()
    return fig

plot_waveform(first_waveform_entry, 0)
plt.show()


## Multi-Hit Event Visualization

In [ ]:
def plot_event_waveforms(event_idx, max_hits=8):
    if event_idx is None: return None
    tree.GetEntry(event_idx)
    if tree.GetBranch('has_sampic_event') and not bool(tree.has_sampic_event):
        print(f'Entry {event_idx} has no waveform bank')
        return None
    event = tree.sampic_event
    num_hits = min(len(event.hits), max_hits)
    if num_hits == 0:
        return None
    fig, axes = plt.subplots(num_hits, 1, figsize=(12, 3.2 * num_hits), squeeze=False)
    for hit_idx, (ax, hit) in enumerate(zip(axes.flat, event.hits)):
        wf = hit_waveform(hit)
        ax.plot(wf, linewidth=1.2)
        ax.set_ylabel(UNIT)
        ax.set_title(f'Hit {hit_idx}: channel {hit.channel}, amplitude {hit_amplitude(hit):.1f}')
    axes.flat[-1].set_xlabel('Sample index')
    fig.suptitle(f'Tree entry {event_idx}')
    plt.tight_layout()
    return fig

plot_event_waveforms(first_waveform_entry)
plt.show()


## Statistical Analysis

In [ ]:
n_events = tree.GetEntries()
baselines, amplitudes, peaks = [], [], []
channels, tot_values = [], []
first_cell_timestamps, fe_board_indices = [], []
num_hits_per_trigger = []

for i in range(n_events):
    tree.GetEntry(i)
    has_event = not tree.GetBranch('has_sampic_event') or bool(tree.has_sampic_event)
    if not has_event:
        num_hits_per_trigger.append(0)
        continue
    event = tree.sampic_event
    num_hits_per_trigger.append(len(event.hits))
    for hit in event.hits:
        baselines.append(hit_baseline(hit))
        amplitudes.append(hit_amplitude(hit))
        peaks.append(hit_peak(hit))
        channels.append(hit.channel)
        tot_values.append(hit.tot_value)
        first_cell_timestamps.append(hit.first_cell_timestamp)
        fe_board_indices.append(hit.fe_board_index)

baselines = np.asarray(baselines)
amplitudes = np.asarray(amplitudes)
peaks = np.asarray(peaks)
channels = np.asarray(channels)
tot_values = np.asarray(tot_values)
first_cell_timestamps = np.asarray(first_cell_timestamps)
num_hits_per_event = np.asarray(num_hits_per_trigger)
fe_board_indices = np.asarray(fe_board_indices)
print(f'Collected {len(amplitudes)} hits from {len(waveform_entries)} waveform entries')


## Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# --- Channel occupancy ---
axes[0, 0].hist(channels, bins=64, range=(0, 64), edgecolor='black')
axes[0, 0].set_xlabel('Channel')
axes[0, 0].set_ylabel('Hits')
axes[0, 0].set_title('Channel Occupancy')

# --- Frontend board index occupancy ---
if fe_board_indices.size > 0:
    bins = np.arange(fe_board_indices.min(), fe_board_indices.max() + 2)
    axes[0, 1].hist(fe_board_indices, bins=bins, edgecolor='black')
axes[0, 1].set_xlabel('FE Board Index')
axes[0, 1].set_ylabel('Hits')
axes[0, 1].set_title('Frontend Board Occupancy')

# --- First cell timestamp distribution ---
axes[1, 0].hist(first_cell_timestamps, bins=100, edgecolor='black')
axes[1, 0].set_xlabel('First Cell Timestamp [ns]')
axes[1, 0].set_ylabel('Counts')
axes[1, 0].set_title('First Cell Timestamp Distribution')
axes[1, 0].set_yscale('log')

# --- Number of hits per event ---
axes[1, 1].hist(num_hits_per_event, bins=50, edgecolor='black')
axes[1, 1].set_xlabel('Number of Hits per Event')
axes[1, 1].set_ylabel('Events')
axes[1, 1].set_title('Event Hit Multiplicity')
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()


## Peak vs Amplitude Correlation

In [ ]:
if amplitudes.size:
    fig, ax = plt.subplots(figsize=(10, 8))
    h = ax.hist2d(amplitudes, peaks, bins=100, cmap='viridis', cmin=1,
                  norm=plt.matplotlib.colors.LogNorm())
    plt.colorbar(h[3], ax=ax, label='Counts')
    ax.set_xlabel(f'Peak-to-peak amplitude [{UNIT}]')
    ax.set_ylabel(f'Waveform maximum [{UNIT}]')
    ax.set_title('Peak vs amplitude correlation')
    plt.tight_layout()
    plt.show()
else:
    print('No hits available for the correlation plot')


## Average Waveform Shape

In [ ]:
all_waveforms = []
for i in waveform_entries:
    tree.GetEntry(i)
    for hit in tree.sampic_event.hits:
        wf = hit_waveform(hit)
        if wf.size:
            all_waveforms.append(wf - hit_baseline(hit))

if all_waveforms:
    min_len = min(map(len, all_waveforms))
    stacked = np.stack([wf[:min_len] for wf in all_waveforms])
    mean = stacked.mean(axis=0)
    std = stacked.std(axis=0)
    x = np.arange(min_len)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(x, mean, label='Average')
    ax.fill_between(x, mean - std, mean + std, alpha=0.3, label='±1σ')
    ax.set_xlabel('Sample index')
    ax.set_ylabel(f'Baseline-subtracted amplitude [{UNIT}]')
    ax.set_title(f'Average waveform shape ({len(all_waveforms)} hits)')
    ax.legend()
    plt.show()
else:
    print('No waveforms found')
